# Lab 02 — Gradient-Based and Blind Optimization

**Solved notebook.** Every task is solved and explained: a *How we got there* cell before each function gives the
reasoning, and an *Answers* cell after each experiment answers the guide's questions with the numbers this notebook
produces. Try the tasks in `lab02_optimization.ipynb` first, then compare.

| Part | Task | Time |
|:--|:--|:-:|
| A | Gradient descent: by hand and with JAX, on P1–P3 | 30 min |
| B | A genetic algorithm from scratch, on P1–P3 | 40 min |
| C | pyBlindOpt: initialization methods and stronger optimizers | 25 min |
| D | Newton's method with `jax.hessian` | 15 min |
| E | Challenge: a classifier trained on the 0/1 loss | home |

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pyBlindOpt as pbo
from lab02_utils import (
    PROBLEMS,
    Counted,
    Recorder,
    best_so_far,
    make_classification,
    plot_boxes,
    plot_convergence,
    plot_fit,
    plot_landscape,
    plot_paths,
    plot_population,
)
from pyBlindOpt import init, utils

plt.rcParams["figure.figsize"] = (9, 4)

## The three problems

| | loss $\mathcal{L}(\theta)$ | parameters $\theta$ | character |
|:--|:--|:--|:--|
| **P1** | $\frac1N\sum_i (w x_i + b - y_i)^2$ | $(w, b) \in [-5,5]^2$ | convex, elongated |
| **P2** | $(1-x)^2 + 100\,(y-x^2)^2$ | $(x, y) \in [-2,2]\times[-1,3]$ | curved valley |
| **P3** | $\frac1N\sum_i (\sin(\omega x_i + \varphi) - y_i)^2$ | $(\omega, \varphi) \in [0.1,4]\times[-\pi,\pi]$ | multimodal |

Each `PROBLEMS[key]` has `.loss` (JAX), `.numpy_loss`, `.bounds`, `.start`, `.optimum` and, for P1 and P3, `.data`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, problem in zip(axes, PROBLEMS.values()):
    plot_landscape(problem, ax)
    ax.plot(*problem.start, "wo", mec="k", ms=8)
    ax.set_title(f"{problem.name}\nwhite dot: GD start, red star: optimum", fontsize=10)
plt.tight_layout()
plt.show()

## Part A — Gradient descent

### A1. The gradient of P1 by hand

Derive $\partial\mathcal{L}/\partial w$ and $\partial\mathcal{L}/\partial b$ for P1 (guide, task A1) and implement
them. The next cell compares your result with `jax.grad`.

### How we got there (A1)

Write the residual of point $i$ as $r_i = w x_i + b - y_i$, so $\mathcal{L} = \frac1N\sum_i r_i^2$. The chain rule gives
$\frac{\partial r_i^2}{\partial w} = 2 r_i \frac{\partial r_i}{\partial w} = 2 r_i x_i$ and $\frac{\partial r_i^2}{\partial b} = 2 r_i$, so

$$\frac{\partial\mathcal{L}}{\partial w} = \frac2N\sum_i r_i\,x_i, \qquad \frac{\partial\mathcal{L}}{\partial b} = \frac2N\sum_i r_i$$

In NumPy the residuals of all points are one vector, `r = w * x + b - y`, and each sum divided by $N$ is a `np.mean`.

In [ ]:
data_p1 = PROBLEMS["P1"].data
assert data_p1 is not None
x_p1, y_p1 = data_p1


def grad_p1_manual(theta) -> np.ndarray:
    w, b = theta
    r = w * x_p1 + b - y_p1
    return np.array([2 * np.mean(r * x_p1), 2 * np.mean(r)])

In [ ]:
grad_p1_jax = jax.grad(PROBLEMS["P1"].loss)
for theta in [np.array([0.0, 0.0]), np.array([-4.0, 4.0]), np.array([1.5, -2.0])]:
    print(theta, "manual:", np.round(grad_p1_manual(theta), 6), " jax:", np.round(np.asarray(grad_p1_jax(theta)), 6))

### Answers (A1)

1–2. The manual gradient matches `jax.grad` to every printed digit at all three points.

3. Setting both derivatives to zero gives $\sum_i r_i x_i = 0$ and $\sum_i r_i = 0$, i.e. two **linear** equations in
$(w, b)$ (the *normal equations*):

$$w\sum_i x_i^2 + b\sum_i x_i = \sum_i x_i y_i, \qquad w\sum_i x_i + b\,N = \sum_i y_i$$

A linear model with a squared loss has a quadratic loss, so its gradient is linear in $\theta$ and the minimum is the
solution of a $2\times2$ linear system: $w = 1.4644$, $b = -1.9390$ (`np.linalg.lstsq`, stored in
`PROBLEMS["P1"].optimum`). No iteration is needed. Gradient descent is for the models where no such closed form exists.

### A2. Gradient descent

$\theta_{k+1} = \theta_k - \eta\,\nabla\mathcal{L}(\theta_k)$. Return the **whole path**: an array of shape
`(n_steps + 1, 2)` whose first row is `theta0`.

### How we got there (A2)

The update is one line, $\theta \leftarrow \theta - \eta\,\nabla\mathcal{L}(\theta)$. We keep every iterate in a list,
because the plots need the whole path, and stack it into an array at the end. `grad` may be a JAX function, so its
output is converted with `np.asarray` to keep the path in NumPy.

In [ ]:
def gradient_descent(grad, theta0, lr, n_steps) -> np.ndarray:
    theta = np.asarray(theta0, dtype=float)
    path = [theta]
    for _ in range(n_steps):
        theta = theta - lr * np.asarray(grad(theta))
        path.append(theta)
    return np.stack(path)

### A3. Learning rates on P1, P2 and P3

Three learning rates per problem, from the problem's start point. Change them and answer the guide's questions.

In [ ]:
LEARNING_RATES = {"P1": [0.02, 0.2, 0.25], "P2": [2e-4, 1e-3, 2e-3], "P3": [0.01, 0.05, 0.2]}
N_STEPS = {"P1": 300, "P2": 3000, "P3": 1000}

gd_results = {}
for key, problem in PROBLEMS.items():
    grad = jax.jit(jax.grad(problem.loss))
    paths = {f"lr = {lr}": gradient_descent(grad, problem.start, lr, N_STEPS[key]) for lr in LEARNING_RATES[key]}
    finite = {k: p for k, p in paths.items() if np.all(np.isfinite(p)) and np.all(np.abs(p) < 1e3)}
    for label in paths.keys() - finite.keys():
        print(f"{key}, {label}: diverged")
    plot_paths(problem, finite, "- gradient descent")
    gd_results[key] = min(finite.values(), key=lambda p: problem.numpy_loss(p[-1]))[-1]
plot_fit(PROBLEMS["P1"], {"GD": gd_results["P1"]}, "- GD result")
plot_fit(PROBLEMS["P3"], {"GD": gd_results["P3"]}, "- GD result")

### Answers (A3)

1. **P1 zig-zags** because its loss surface is a long, thin valley: the Hessian eigenvalues are $0.36$ and $8.24$
   (condition number $\approx 23$). The gradient is perpendicular to the contour lines, not aimed at the minimum, and the
   steep direction limits the step: GD is stable only for $\eta < 2/8.24 = 0.243$.
   * $\eta = 0.25 > 0.243$ **diverges** (the loss reaches $10^{16}$ within 300 steps), exactly as predicted.
   * $\eta = 0.2$ gets within $10^{-6}$ of the minimum in 110 steps; $\eta = 0.24$, right at the edge, needs 299 steps,
     because along the steep direction the error is multiplied by $|1 - 0.24 \cdot 8.24| = 0.98$ per step.
2. **P2** (from $(-1.5, 2.5)$, 20000 steps): $\eta = 3\cdot10^{-3}$ diverges. $\eta = 2.5\cdot10^{-3}$ does not blow up but
   bounces across the valley and stalls at a loss of $0.18$. $\eta = 2\cdot10^{-3}$ is the largest that converges: it is
   within $10^{-3}$ after 239 steps, then hovers there. With $\eta = 10^{-3}$ the same target takes **7294 steps**
   ($\eta = 1.5\cdot10^{-3}$: 3292). The limit again comes from the steep curvature *across* the valley.
3. **P3** from $(1, 0)$: GD stops in a **local minimum** with a wrong frequency ($\omega \approx 1.1$; the loss is about
   $0.9$ against $0.037$ at the global minimum). The fitted sine matches part of the data and averages the rest. To
   reach $\omega \approx 1.7$ it would have to climb over the barriers between frequencies, and gradient descent only
   goes downhill.

### A4. Multi-start on P3

Run gradient descent from `n_starts` random points of the box (uniform) and return the final points, shape
`(n_starts, 2)`.

### How we got there (A4)

`rng.uniform(low, high, (n, 2))` broadcasts the per-coordinate bounds, so one call draws all the starts inside the box.
Each start runs the `gradient_descent` of A2, and we keep only the last point of each path.

In [ ]:
def multistart_gd(problem, n_starts, lr, n_steps, seed=0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    grad = jax.jit(jax.grad(problem.loss))
    starts = rng.uniform(problem.bounds[:, 0], problem.bounds[:, 1], (n_starts, 2))
    return np.stack([gradient_descent(grad, s, lr, n_steps)[-1] for s in starts])

In [ ]:
problem = PROBLEMS["P3"]
finals = multistart_gd(problem, n_starts=30, lr=0.05, n_steps=500)
hits = np.linalg.norm(finals - problem.optimum, axis=1) < 0.05
ax = plot_landscape(problem)
ax.scatter(finals[:, 0], finals[:, 1], c=np.where(hits, "tab:green", "tab:red"), edgecolor="k", zorder=3)
ax.set_title(f"P3: {hits.sum()} of {len(finals)} starts reach the global minimum ({30 * 500} gradient evaluations)")
plt.show()

### Answers (A4)

Only **4 of 30** starts (13%) reach the global minimum, and the multi-start cost $30 \times 500 = 15000$ gradient
evaluations. The global basin is a narrow diagonal band around $\omega \approx 1.7$; most random starts lie in the basin
of another frequency. Multi-start is the simplest global strategy, but it wastes most of its budget in the wrong basins.

## Part B — A genetic algorithm from scratch

Read the design considerations in the guide (Part B) first. Every function receives a NumPy random generator `rng`,
so a run is reproducible from its seed. `bounds` has shape `(d, 2)`: column 0 the lower, column 1 the upper limits.

### How we got there (B1, the operators)

* **`init_population`**: `rng.uniform(bounds[:, 0], bounds[:, 1], (n_pop, d))` draws every gene inside its own interval.
* **`tournament`**: `rng.choice(len(pop), size=k, replace=False)` picks $k$ distinct contestants, and `np.argmin` over their
  losses picks the winner. We return a **copy**, so later changes to the child never modify the population.
* **`blend_crossover`**: with `lo`, `hi` the per-gene min and max of the parents and `d = hi - lo`, each child is
  `rng.uniform(lo - alpha * d, hi + alpha * d)`. That is two independent draws, so the two children differ.
* **`gaussian_mutation`**: a Boolean mask `rng.random(shape) < rate` selects the genes to mutate. The noise has standard
  deviation `sigma_frac * (high - low)`, so the step scales with each box side. `np.clip` puts the child back in the box.

In [ ]:
def init_population(bounds, n_pop, rng) -> np.ndarray:
    """n_pop points drawn uniformly inside the box, shape (n_pop, d)."""
    return rng.uniform(bounds[:, 0], bounds[:, 1], (n_pop, bounds.shape[0]))


def tournament(pop, fitness, k, rng) -> np.ndarray:
    """Pick k individuals at random and return a copy of the best one (lowest fitness)."""
    idx = rng.choice(len(pop), size=k, replace=False)
    return pop[idx[np.argmin(fitness[idx])]].copy()


def blend_crossover(p1, p2, alpha, rng) -> tuple[np.ndarray, np.ndarray]:
    """BLX-alpha: each gene of each child is uniform in [lo - alpha*d, hi + alpha*d], lo/hi the parents' genes."""
    lo, hi = np.minimum(p1, p2), np.maximum(p1, p2)
    d = hi - lo
    return rng.uniform(lo - alpha * d, hi + alpha * d), rng.uniform(lo - alpha * d, hi + alpha * d)


def gaussian_mutation(child, bounds, rate, sigma_frac, rng) -> np.ndarray:
    """With probability `rate` per gene, add N(0, (sigma_frac * box width)^2); then clip to the box."""
    width = bounds[:, 1] - bounds[:, 0]
    mask = rng.random(child.shape) < rate
    child = child + mask * rng.normal(0, sigma_frac * width)
    return np.clip(child, bounds[:, 0], bounds[:, 1])

### B1. The GA loop

1. Initialize and evaluate the population.
2. Each generation: keep the `n_elite` best as they are; fill the rest of the new population with children
   (two tournaments → crossover → mutation of each child); evaluate the new population.
3. Track the best individual ever seen.

Return `(best_theta, best_loss, history)` where `history` is the list of populations, one per generation
(including generation 0). Evaluate a whole population at once: `loss(pop)` returns one value per row.

### How we got there (B1, the loop)

* **Elitism first:** `np.argsort(fitness)[:n_elite]` copies the best individuals unchanged.
* **Children until full:** each pair of tournaments produces two children, so the list can overshoot by one when
  `n_pop - n_elite` is odd. We keep `children[:n_pop]`.
* **One evaluation per generation:** `loss(pop)` evaluates all rows at once (the problems are vectorized).
* **Best ever:** with elitism the best never gets lost, but we still track it explicitly, so the function stays correct
  when `n_elite = 0` (B3).
* **Our choices and why:** $N = 30$ (plenty for 2 parameters); $k = 3$ (mild pressure); $\alpha = 0.5$ (children can
  leave the parents' box, so the population does not shrink too fast); `rate = 0.5` (about one gene of two per child);
  $\sigma$ = 10% of the box width (large enough to jump between nearby basins, small enough to refine); 2 elites; 60
  generations, i.e. $30 \times 61 = 1830$ evaluations.

In [ ]:
def genetic_algorithm(
    loss, bounds, n_pop=30, n_gen=60, k=3, alpha=0.5, rate=0.5, sigma_frac=0.1, n_elite=2, seed=0
) -> tuple[np.ndarray, float, list[np.ndarray]]:
    rng = np.random.default_rng(seed)
    pop = init_population(bounds, n_pop, rng)
    fitness = loss(pop)
    history = [pop.copy()]
    best = np.argmin(fitness)
    best_theta, best_loss = pop[best].copy(), float(fitness[best])
    for _ in range(n_gen):
        order = np.argsort(fitness)
        children = [pop[i].copy() for i in order[:n_elite]]
        while len(children) < n_pop:
            p1, p2 = tournament(pop, fitness, k, rng), tournament(pop, fitness, k, rng)
            for child in blend_crossover(p1, p2, alpha, rng):
                children.append(gaussian_mutation(child, bounds, rate, sigma_frac, rng))
        pop = np.stack(children[:n_pop])
        fitness = loss(pop)
        history.append(pop.copy())
        best = np.argmin(fitness)
        if fitness[best] < best_loss:
            best_theta, best_loss = pop[best].copy(), float(fitness[best])
    return best_theta, best_loss, history

### B2. Run it on the three problems

Budget: 30 individuals × 61 generations = 1830 evaluations of the loss (and **no** gradient).

In [ ]:
ga_results = {}
for key, problem in PROBLEMS.items():
    theta, value, history = genetic_algorithm(problem.numpy_loss, problem.bounds, seed=1)
    ga_results[key] = theta
    print(f"{key}: GA best {np.round(theta, 4)}, loss {value:.5f}  |  GD best loss "
          f"{float(problem.numpy_loss(gd_results[key])):.5f}  |  optimum loss {float(problem.numpy_loss(problem.optimum)):.5f}")
    plot_population(problem, history, epochs=(0, 3, 10, -1), title="- your GA")
plot_fit(PROBLEMS["P3"], {"GD": gd_results["P3"], "GA": ga_results["P3"]}, "- GD vs. GA")

### Answers (B2)

1. **P1:** the GA reaches a loss of $0.04580$ against $0.04578$ for GD (the optimum is $0.04578$). GD is **more precise**
   (it converges to machine precision) and **cheaper** (300 gradient steps, each worth a few loss evaluations, against
   1830 evaluations). The GA finds the right region quickly, but refining by random mutations is slow. On a smooth
   convex problem the gradient wins.
2. **P3:** GD ends in a local minimum (loss $0.896$, wrong frequency), while the GA reaches $0.03722$, the global minimum.
   The initial population covers the whole box, so a few individuals already start in the global basin. Selection
   copies them, and crossover and mutation explore around them: after 3 generations most of the population is in the
   right band (see the snapshots). The population searches **globally**; gradient descent only follows its own basin.

### B3. Selection pressure and mutation strength

10 seeds per setting on P3, with a **small budget** (15 generations) so the differences show. `k = 1` is random
selection (no pressure); `sigma_frac` sets the mutation step.

In [ ]:
settings = {
    "k=1": dict(k=1),
    "k=2": dict(k=2),
    "k=3 (default)": dict(k=3),
    "k=8": dict(k=8),
    "sigma=0.01": dict(sigma_frac=0.01),
    "sigma=0.3": dict(sigma_frac=0.3),
    "no elitism": dict(n_elite=0),
}
problem = PROBLEMS["P3"]
results = {name: [genetic_algorithm(problem.numpy_loss, problem.bounds, n_gen=15, seed=s, **kw)[1] for s in range(10)]
           for name, kw in settings.items()}
plot_boxes(results, title="P3: final loss over 10 seeds")

### Answers (B3)

Median (worst) final loss over 10 seeds with 15 generations. The optimum is $0.0372$.

| setting | median | worst | why |
|:--|--:|--:|:--|
| $k = 1$ | 0.0571 | 0.1175 | no selection pressure: a random walk, it barely improves |
| $k = 2$ | 0.0404 | 0.0606 | mild pressure, still slow |
| $k = 3$ | 0.0374 | 0.0496 | default: fast and reliable |
| $k = 8$ | 0.0376 | 0.0390 | strong pressure: fastest here, because in 2D the global basin is found in the first generations; in harder problems it risks premature convergence |
| $\sigma = 0.01$ | 0.0373 | 0.0500 | precise, but one seed stays in a wrong basin: small steps cannot jump |
| $\sigma = 0.3$ | 0.0402 | 0.0648 | steps too coarse to refine the answer |
| no elitism | 0.0387 | 0.2352 | the best individual can be lost: two seeds end far from the optimum |

Every parameter trades exploration against exploitation, as described in the guide.

## Part C — pyBlindOpt: initialization and stronger optimizers

### C1. Initial populations

Return an initial population of `n_pop` points for `problem` using one of the methods below (see the guide and
notebook `02_blind_optimization.ipynb`, demo B2):

`"Random"`, `"LHS"`, `"Sobol"`, `"Chaotic"` (samplers in `pyBlindOpt.utils`), `"OBL"`, `"QOBL"`, `"OBLESA"`
(strategies in `pyBlindOpt.init`, which evaluate the loss to choose the points).

### How we got there (C1)

The samplers only need the generator, `init.get_initial_population(n_pop, bounds, Sampler(rng))`. The strategies also
need the loss, because they evaluate candidates to choose the start: they get a `RandomSampler(rng)` as the base
population and `seed=rng`. A dictionary maps each sampler name to its class, so there is only one code path per family.

In [ ]:
INIT_METHODS = ["Random", "LHS", "Sobol", "Chaotic", "OBL", "QOBL", "OBLESA"]


def make_population(kind, problem, n_pop, rng) -> np.ndarray:
    samplers = {"Random": utils.RandomSampler, "LHS": utils.HLCSampler, "Sobol": utils.SobolSampler,
                "Chaotic": utils.ChaoticSampler}
    if kind in samplers:
        return init.get_initial_population(n_pop, problem.bounds, samplers[kind](rng))
    base = utils.RandomSampler(rng)
    f = problem.numpy_loss
    if kind == "OBL":
        return init.opposition_based(f, problem.bounds, population=base, n_pop=n_pop, seed=rng)
    if kind == "QOBL":
        return init.quasi_opposition_based(f, problem.bounds, population=base, n_pop=n_pop, seed=rng)
    if kind == "OBLESA":
        return init.oblesa(f, problem.bounds, population=base, n_pop=n_pop, seed=rng)
    raise ValueError(kind)

In [ ]:
problem = PROBLEMS["P3"]
fig, axes = plt.subplots(1, len(INIT_METHODS), figsize=(3.2 * len(INIT_METHODS), 3.2))
for ax, kind in zip(axes, INIT_METHODS):
    pop = make_population(kind, problem, 20, np.random.default_rng(0))
    plot_landscape(problem, ax)
    ax.scatter(pop[:, 0], pop[:, 1], s=14, color="w", edgecolor="k")
    ax.set(title=f"{kind}\nbest {problem.numpy_loss(pop).min():.3f}", xlabel="", ylabel="")
plt.tight_layout()
plt.show()

### Answers (C1)

Best initial loss on P3 (seed 0, 20 points; the optimum is 0.037): Random 0.632, LHS 0.401, Sobol 0.306, Chaotic 0.762,
OBL 0.244, QOBL **0.058**, OBLESA **0.058**.

* **Samplers** (Random, LHS, Sobol, Chaotic) cost nothing: they only place points. **Strategies** evaluate candidates
  before the search starts: OBL and QOBL evaluate $2N = 40$ points, and OBLESA about $3N = 60$.
* **Why QOBL and OBLESA start so much better:** the quasi-opposite point is drawn between the box centre and the
  opposite. The centre of P3's box is $(2.05, 0)$ and the optimum $(1.705, 0.813)$ lies close to it, so pulling points
  towards the centre helps. This is the centre bias from the lecture: on a problem whose optimum is near a corner, the
  advantage shrinks. OBLESA's default opposition is the quasi one, hence the same best point.

### C2. Stronger optimizers

Run one pyBlindOpt optimizer by name — `"GA"`, `"DE"`, `"SHADE"` (DE with `variant="current-to-pbest/1/bin"`,
`policy="shade"`) or `"EGWO"` — from a given initial population, and return the best loss. Use `n_iter` epochs and
pass `seed` to the optimizer. Every optimizer then has the same budget: `len(population)` × (`n_iter` + 1).

### How we got there (C2)

All four optimizers share the pyBlindOpt signature `f(objective, bounds, population=..., n_iter=..., seed=...)` and
return `(best_x, best_f)`. We build the keyword arguments once and dispatch on the name. SHADE is DE with a different
variant and policy. Passing the same `population` to every optimizer gives them the same start and the same budget.

In [ ]:
def run_library(name, problem, population, n_iter, seed) -> float:
    kwargs = dict(population=population, n_iter=n_iter, seed=seed)
    f, b = problem.numpy_loss, problem.bounds
    if name == "GA":
        return float(pbo.genetic_algorithm(f, b, **kwargs)[1])
    if name == "DE":
        return float(pbo.differential_evolution(f, b, **kwargs)[1])
    if name == "SHADE":
        return float(pbo.differential_evolution(f, b, variant="current-to-pbest/1/bin", policy="shade", **kwargs)[1])
    if name == "EGWO":
        return float(pbo.enhanced_grey_wolf_optimization(f, b, **kwargs)[1])
    raise ValueError(name)

Budget per run: 20 individuals × 31 epochs = 620 evaluations, 15 seeds. First the optimizers (random
initialization, plus your GA with the same budget), then the initialization methods (with DE).

In [ ]:
N_POP, N_ITER, SEEDS = 20, 30, range(15)
for key, problem in PROBLEMS.items():
    optimum = float(problem.numpy_loss(problem.optimum))
    res = {name: [run_library(name, problem, make_population("Random", problem, N_POP, np.random.default_rng(s)),
                              N_ITER, s) - optimum for s in SEEDS]
           for name in ["GA", "DE", "SHADE", "EGWO"]}
    res["your GA"] = [genetic_algorithm(problem.numpy_loss, problem.bounds, n_pop=N_POP, n_gen=N_ITER, seed=s)[1]
                      - optimum for s in SEEDS]
    plot_boxes(res, title=f"{problem.name}: optimizers (620 evaluations, 15 seeds)", ylabel="final loss - optimum")

In [ ]:
for key, problem in PROBLEMS.items():
    optimum = float(problem.numpy_loss(problem.optimum))
    res = {kind: [run_library("DE", problem, make_population(kind, problem, N_POP, np.random.default_rng(s)),
                              N_ITER, s) - optimum for s in SEEDS]
           for kind in INIT_METHODS}
    plot_boxes(res, title=f"{problem.name}: DE with each initialization (15 seeds)", ylabel="final loss - optimum")

### Answers (C2)

Median of loss $-$ optimum over 15 seeds (620 evaluations each):

| | GA | DE | SHADE | EGWO | our GA |
|:--|--:|--:|--:|--:|--:|
| P1 | 4.7e-05 | **5.9e-12** | 9.6e-06 | 2.7e-05 | 4.3e-04 |
| P2 | 1.4e-02 | **2.3e-10** | 4.7e-03 | 5.3e-03 | 8.4e-03 |
| P3 | 5.2e-05 | **8.3e-12** | 1.7e-04 | 4.9e-05 | 2.8e-05 |

1. **Classic DE (`best/1`) wins all three problems here.** In 2D with 620 evaluations its greedy strategy, always building
   the mutant from the best point, converges fastest. This does *not* contradict No Free Lunch: in the lecture (demo B4,
   10-dimensional Rastrigin) the same `best/1` was among the weaker population methods. Small, easy problems reward
   greed, and SHADE's adaptation needs more generations to pay off.
2. **Our GA and pyBlindOpt's GA are in the same range:** ours is worse on P1 ($4.3\cdot10^{-4}$ against $4.7\cdot10^{-5}$) and
   better on P2 and P3. The library uses a different mutation (polynomial), a mutation rate of $1/d$ and 10% elitism: the
   design choices of Part B, made differently.
3. **Initialization barely changes DE here:** every method ends between $10^{-12}$ and $10^{-9}$. QOBL and OBLESA give the
   lowest medians on P1 and P3 ($1.3$–$3\cdot10^{-12}$ against $6$–$8\cdot10^{-12}$ for random), and on P2 there is no
   consistent gain. With 620 evaluations in 2D, DE converges from any start. Initialization matters most with small
   budgets and many dimensions (lecture, demo B2).

### C3. Watch one run

The class interface keeps the optimizer object, so a `Recorder` callback can store every population.

In [ ]:
problem = PROBLEMS["P3"]
opt = pbo.DifferentialEvolution(problem.numpy_loss, problem.bounds, n_pop=20, n_iter=30, seed=3)
rec = Recorder.attach(opt)
opt.optimize()
plot_population(problem, rec.pops, epochs=(0, 3, 10, -1), title="- pyBlindOpt DE")
plot_convergence({"DE": best_so_far(rec.pops, problem)}, title="P3: best loss so far")

### Answers (C3)

DE's population collapses from the whole box onto the global band within about 10 epochs, then onto the optimum. Its
steps are differences between members, so they shrink automatically as the population contracts. The GA's population
(Part B) stays a cloud whose size is set by the mutation $\sigma$: it keeps exploring around the optimum, which is why it
is less precise.

## Part D — Newton's method

$\theta_{k+1} = \theta_k - (H + \lambda I)^{-1}\,\nabla\mathcal{L}(\theta_k)$, with $H$ from `jax.hessian`.
$\lambda = 0$ is pure Newton; $\lambda > 0$ ("damping") blends it with a small gradient step. Return the path.

### How we got there (D1)

The second-order Taylor model around $\theta_k$ is $\mathcal{L}(\theta_k + \delta) \approx \mathcal{L} + g^\top\delta + \frac12
\delta^\top H \delta$. Setting its gradient $g + H\delta$ to zero gives the step $\delta = -H^{-1}g$. We compute it with
`np.linalg.solve(H, g)`, which is cheaper and more accurate than inverting $H$. Damping adds $\lambda I$ to $H$: large
$\lambda$ makes the step $\approx -g/\lambda$, a small gradient step, and $\lambda = 0$ is pure Newton.

In [ ]:
def newton(problem, theta0, n_steps, damping=0.0) -> np.ndarray:
    grad = jax.jit(jax.grad(problem.loss))
    hess = jax.jit(jax.hessian(problem.loss))
    theta = np.asarray(theta0, dtype=float)
    path = [theta]
    for _ in range(n_steps):
        H = np.asarray(hess(theta)) + damping * np.eye(theta.size)
        theta = theta - np.linalg.solve(H, np.asarray(grad(theta)))
        path.append(theta)
    return np.stack(path)

In [ ]:
for key, problem in PROBLEMS.items():
    grad = jax.jit(jax.grad(problem.loss))
    lr = LEARNING_RATES[key][1]
    paths = {
        f"GD lr={lr}, 20 steps": gradient_descent(grad, problem.start, lr, 20),
        "Newton, 20 steps": newton(problem, problem.start, 20),
        "damped Newton (lambda=1), 20 steps": newton(problem, problem.start, 20, damping=1.0),
    }
    for label, p in paths.items():
        print(f"{key} {label:36s} final theta {np.round(p[-1], 4)}, loss {float(problem.numpy_loss(p[-1])):.5f}")
    plot_paths(problem, paths, "- GD vs. Newton")

### Answers (D)

1. P1's loss is exactly quadratic, so the second-order Taylor model *is* the loss: the first Newton step lands on the
   least-squares solution $(1.4644, -1.9390)$.
2. P2: Newton is below $10^{-3}$ after **5 steps** and below $10^{-10}$ after 6, against 7294 GD steps at $\eta = 10^{-3}$
   (Part A). The curvature gives both the direction along the curved valley and the right step length.
3. P3: from $(1, 0)$ pure Newton converges to $(1.013, 0.081)$, where the Hessian eigenvalues are $-0.111$ and $20.6$:
   one negative, one positive, so it is a **saddle point**. Newton solves $\nabla\mathcal{L} = 0$ and cannot tell minima
   from saddles. Damping makes the step closer to a gradient step, so it goes downhill again (loss $0.984$), but it
   stays in the same poor basin. Curvature does not fix a bad start; that is the job of the global (blind) search.
4. With $d = 10^6$ weights the Hessian has $10^{12}$ entries (8 TB in float64), and each solve costs $O(d^3)$. Neural
   network losses are also full of saddle points, which attract pure Newton. Deep learning therefore uses first-order
   methods with a cheap diagonal rescaling (RMSProp, Adam).

## Part E — Challenge: optimizing what we actually care about

A linear classifier predicts $\hat y = [\,w^\top x + b > 0\,]$. We want the lowest **error rate** (0/1 loss), but
its gradient is zero almost everywhere. Two routes (guide, Part E):

1. **Surrogate + gradient:** minimize the smooth logistic loss with gradient descent (what logistic regression does).
2. **Blind:** minimize the error rate itself with a population method.

$\theta = (w_1, \dots, w_6, b) \in [-5, 5]^7$.

### How we got there (E1–E2)

**Error rate for one point or a population.** `np.atleast_2d` turns one $\theta$ of shape `(D,)` into `(1, D)`, so a
single code path handles both. The weights are `theta[:, :-1]` and the biases `theta[:, -1]`. Then
`X @ theta[:, :-1].T + theta[:, -1]` computes the score of every data point under every candidate, shape
`(N, n_pop)`, in **one matrix product**. A prediction is `score > 0`; comparing with `y[:, None]` and averaging over
axis 0 gives one error rate per candidate. For a single $\theta$ we return a plain float.

**Logistic loss.** With $z = w^\top x + b$ and $s(z) = 1/(1+e^{-z})$ the probability of class 1, the binary cross-entropy is

$$-\big[y \log s(z) + (1-y)\log(1-s(z))\big] = \log(1 + e^{z}) - y\,z$$

(use $\log s(z) = z - \log(1+e^z)$ and $\log(1 - s(z)) = -\log(1+e^z)$). `jnp.logaddexp(0, z)` computes
$\log(e^0 + e^z) = \log(1+e^z)$ without overflowing for large $z$. It is written with `jax.numpy`, so `jax.grad`
derives its gradient for the gradient descent of Part A.

In [ ]:
X_train, y_train, X_test, y_test = make_classification(seed=0)
D = X_train.shape[1] + 1
BOUNDS_E = np.array([[-5.0, 5.0]] * D)
print("train", X_train.shape, "test", X_test.shape, "class balance", y_train.mean().round(2))


def error_rate(theta, X, y) -> float | np.ndarray:
    theta = np.atleast_2d(theta)
    pred = (X @ theta[:, :-1].T + theta[:, -1] > 0).astype(int)  # (n, n_pop)
    err = np.mean(pred != y[:, None], axis=0)
    return err if err.size > 1 else float(err[0])


def logistic_loss(theta, X, y) -> jax.Array:
    z = X @ theta[:-1] + theta[-1]
    return jnp.mean(jnp.logaddexp(0.0, z) - y * z)  # = -[y log s(z) + (1-y) log(1-s(z))]

In [ ]:
theta0 = np.zeros(D)
zero_one = lambda t: jnp.mean((X_train @ t[:-1] + t[-1] > 0) != y_train)  # noqa: E731
print("gradient of the 0/1 loss at a random point:", jax.grad(lambda t: zero_one(t).astype(float))(np.ones(D)))
print("gradient of the logistic loss at 0:       ", np.round(np.asarray(jax.grad(logistic_loss)(theta0, X_train, y_train)), 4))

In [ ]:
SEEDS_E = range(10)
grad_log = jax.jit(jax.grad(logistic_loss))
theta_log = gradient_descent(lambda t: grad_log(t, X_train, y_train), theta0, lr=0.1, n_steps=2000)[-1]
results_e = {"GD on logistic loss": [float(error_rate(theta_log, X_test, y_test))] * len(SEEDS_E)}


def train_error(theta):
    return error_rate(theta, X_train, y_train)


for name, fn, kw in [
    ("DE on 0/1 loss", pbo.differential_evolution, {}),
    ("SHADE on 0/1 loss", pbo.differential_evolution, dict(variant="current-to-pbest/1/bin", policy="shade")),
    ("EGWO on 0/1 loss", pbo.enhanced_grey_wolf_optimization, {}),
]:
    results_e[name] = []
    for s in SEEDS_E:
        counted = Counted(train_error)
        theta, _ = fn(counted, BOUNDS_E, n_pop=40, n_iter=150, seed=s, **kw)
        results_e[name].append(float(error_rate(theta, X_test, y_test)))
results_e["your GA on 0/1 loss"] = [
    float(error_rate(genetic_algorithm(train_error, BOUNDS_E, n_pop=40, n_gen=150, seed=s)[0], X_test, y_test))
    for s in SEEDS_E
]
print(f"evaluations per blind run: {counted.n}")
for name, errs in results_e.items():
    print(f"{name:24s} test error: median {np.median(errs):.3f}, min {np.min(errs):.3f}, max {np.max(errs):.3f}")
plot_boxes(results_e, title="Test error over 10 seeds (label noise 10%)", ylabel="test error", log=False)

### Answers (E)

1–2. See the functions above: one matrix product for the whole population; `logaddexp` for a stable logistic loss.
3. The 0/1 loss is **piecewise constant**: a tiny change of $\theta$ flips no prediction, so the loss does not change,
   and `jax.grad` returns exactly zero. A gradient method gets no signal at all.
4. Test error over 10 seeds:

   | method | median | range |
   |:--|--:|:--|
   | GD on the logistic loss | 0.124 | deterministic |
   | DE on the 0/1 loss | 0.118 | 0.103–0.132 |
   | SHADE on the 0/1 loss | **0.106** | 0.104–0.107 |
   | EGWO on the 0/1 loss | **0.106** | 0.103–0.121 |
   | our GA on the 0/1 loss | 0.107 | 0.103–0.116 |

   **Lower:** the blind methods that optimize the 0/1 loss itself. The logistic loss is a *different* objective, pulled
   by points far from the boundary, including the 10% flipped labels: its training error is 0.14, while SHADE reaches
   0.10 on the same training set. **More stable:** GD is deterministic; among the blind methods SHADE has the smallest
   spread and classic DE the largest. **Cheaper:** GD, with 2000 gradient steps against 6040 evaluations per blind
   run (and 10 runs to measure the spread).
5. $\operatorname{sign}\big(c\,(w^\top x + b)\big) = \operatorname{sign}(w^\top x + b)$ for any $c > 0$, so every point on a
   ray from the origin is the same classifier (tripling SHADE's solution gives the same 0.107 test error). The landscape
   is flat along those rays: large plateaus that the optimizer cannot tell apart. Only the *direction* of $\theta$
   matters, and the box bounds only set an arbitrary scale.